In [1]:
import os
import random
import json
from typing import Dict, List

import numpy as np
import pandas as pd
from tqdm.auto import tqdm

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

from transformers import (
    AutoTokenizer,
    RobertaModel,
    DataCollatorWithPadding,
    get_linear_schedule_with_warmup,
)
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)
if device.type == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))

def set_seed(seed: int = 42) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(42)

Device: cuda
GPU: Tesla T4


In [2]:
TRAIN_PATH = "data/train.csv"
DEV_PATH = "data/dev.csv"

assert os.path.exists(TRAIN_PATH), "train.csv not found in data/ folder."
assert os.path.exists(DEV_PATH), "dev.csv not found in data/ folder."

MODEL_NAME = "roberta-base"
OUTPUT_DIR = "roberta_custom_head_nli"
os.makedirs(OUTPUT_DIR, exist_ok=True)

print("Found:", TRAIN_PATH, DEV_PATH)
print("Model:", MODEL_NAME)
print("Will save model to:", OUTPUT_DIR)

train_df = pd.read_csv(TRAIN_PATH)
dev_df = pd.read_csv(DEV_PATH)

print("Train shape:", train_df.shape)
print("Dev shape:", dev_df.shape)

display(train_df.head())
display(dev_df.head())

Found: data/train.csv data/dev.csv
Model: roberta-base
Will save model to: roberta_custom_head_nli
Train shape: (24432, 3)
Dev shape: (6736, 3)


,premise,hypothesis,label
0,yeah i don't know cut California in half or so...,Yeah. I'm not sure how to make that fit. Maybe...,1
1,actual names will not be used,"For the sake of privacy, actual names are not ...",1
2,The film was directed by Randall Wallace.,The film was directed by Randall Wallace and s...,1
3,"""How d'you know he'll sign me on?""Anse studie...",Anse looked at himself in a cracked mirror.,1
4,In the light of the candles his cheeks looked ...,Drew regarded his best friend and noted that i...,1


,premise,hypothesis,label
0,"By starting at the soft underbelly, the 16,000...","General Nelson A. Miles had 30,000 troops in h...",0
1,"The class had broken into a light sweat, but w...",The class grew more tense as time went on.,1
2,"Samson had his famous haircut here, but he wou...",It was unknown where exactly within the town S...,1
3,A man with a black shirt holds a baby while a ...,A darkly dressed man passes a crying baby to a...,0
4,I know that many of you are interested in addr...,The problems must be addressed,1


In [3]:
def guess_column(df: pd.DataFrame, candidates):
    cols_lower = {c.lower(): c for c in df.columns}
    for cand in candidates:
        if cand in cols_lower:
            return cols_lower[cand]
    return None

premise_col = guess_column(train_df, ["premise", "sentence1", "text1", "s1"])
hypothesis_col = guess_column(train_df, ["hypothesis", "sentence2", "text2", "s2"])
label_col = guess_column(train_df, ["label", "gold_label", "class", "y"])

print("Guessed columns:")
print("  premise_col   =", premise_col)
print("  hypothesis_col=", hypothesis_col)
print("  label_col     =", label_col)

if premise_col is None or hypothesis_col is None or label_col is None:
    raise ValueError(f"Could not confidently detect columns. Columns found: {list(train_df.columns)}")

# Basic cleaning
for df in [train_df, dev_df]:
    df[premise_col] = df[premise_col].fillna("").astype(str).str.strip()
    df[hypothesis_col] = df[hypothesis_col].fillna("").astype(str).str.strip()

# Remove rows with empty premise/hypothesis if any
train_df = train_df[(train_df[premise_col] != "") & (train_df[hypothesis_col] != "")].copy()
dev_df = dev_df[(dev_df[premise_col] != "") & (dev_df[hypothesis_col] != "")].copy()

print("Train label examples:", train_df[label_col].head(10).tolist())
print("Unique train labels:", sorted(train_df[label_col].unique().tolist()))

unique_labels = sorted(train_df[label_col].unique().tolist())
label2id = {lab: i for i, lab in enumerate(unique_labels)}
id2label = {i: lab for lab, i in label2id.items()}

print("label2id:", label2id)

train_df["label_id"] = train_df[label_col].map(label2id)
dev_df["label_id"] = dev_df[label_col].map(label2id)

assert dev_df["label_id"].isna().sum() == 0, "Dev set contains unseen labels."

train_df["label_id"] = train_df["label_id"].astype(int)
dev_df["label_id"] = dev_df["label_id"].astype(int)

num_labels = len(label2id)
print("num_labels:", num_labels)

print("\nTrain label distribution:")
display(train_df[label_col].value_counts())

print("\nDev label distribution:")
display(dev_df[label_col].value_counts())

train_lens = train_df[premise_col].str.split().str.len() + train_df[hypothesis_col].str.split().str.len()
dev_lens = dev_df[premise_col].str.split().str.len() + dev_df[hypothesis_col].str.split().str.len()

print("\nApprox total word length stats (premise+hypothesis):")
print("Train:\n", train_lens.describe(percentiles=[0.5, 0.9, 0.95]).to_string())
print("Dev:\n", dev_lens.describe(percentiles=[0.5, 0.9, 0.95]).to_string())

Guessed columns:
  premise_col   = premise
  hypothesis_col= hypothesis
  label_col     = label
Train label examples: [1, 1, 1, 1, 1, 0, 0, 0, 1, 1]
Unique train labels: [0, 1]
label2id: {0: 0, 1: 1}
num_labels: 2

Train label distribution:


,count
label,
1,12646
0,11783



Dev label distribution:


,count
label,
1,3478
0,3258



Approx total word length stats (premise+hypothesis):
Train:
 count    24429.000000
mean        29.365672
std         14.999346
min          2.000000
50%         27.000000
90%         47.000000
95%         54.000000
max        292.000000
Dev:
 count    6736.000000
mean       29.022417
std        14.079020
min         2.000000
50%        27.000000
90%        47.000000
95%        53.000000
max       147.000000


In [4]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

MAX_LENGTH = 256
BATCH_SIZE = 16
EPOCHS = 3
LEARNING_RATE = 2e-5
WEIGHT_DECAY = 0.01
WARMUP_RATIO = 0.06
MAX_GRAD_NORM = 1.0
DROPOUT = 0.2
HIDDEN_DIM_1 = 256
HIDDEN_DIM_2 = 128

print("Model:", MODEL_NAME)
print("Max length:", MAX_LENGTH)
print("Batch size:", BATCH_SIZE)
print("Epochs:", EPOCHS)
print("LR:", LEARNING_RATE)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/481 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Model: roberta-base
Max length: 256
Batch size: 16
Epochs: 3
LR: 2e-05


In [5]:
class NLIPairDataset(Dataset):
    def __init__(self, df: pd.DataFrame, tokenizer, premise_col: str, hypothesis_col: str, label_col: str, max_length: int):
        self.df = df.reset_index(drop=True)
        self.tokenizer = tokenizer
        self.premise_col = premise_col
        self.hypothesis_col = hypothesis_col
        self.label_col = label_col
        self.max_length = max_length

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx: int):
        row = self.df.iloc[idx]
        prem = row[self.premise_col]
        hyp = row[self.hypothesis_col]
        label = int(row[self.label_col])

        enc = self.tokenizer(
            prem,
            hyp,
            truncation=True,
            max_length=self.max_length,
            padding=False,  # dynamic padding in collator
            return_tensors=None,
        )

        enc["labels"] = label
        return enc

In [6]:
train_ds = NLIPairDataset(
    train_df,
    tokenizer,
    premise_col=premise_col,
    hypothesis_col=hypothesis_col,
    label_col="label_id",
    max_length=MAX_LENGTH
)

dev_ds = NLIPairDataset(
    dev_df,
    tokenizer,
    premise_col=premise_col,
    hypothesis_col=hypothesis_col,
    label_col="label_id",
    max_length=MAX_LENGTH
)

data_collator = DataCollatorWithPadding(tokenizer=tokenizer, padding=True)

train_loader = DataLoader(
    train_ds,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=0,
    collate_fn=data_collator
)

dev_loader = DataLoader(
    dev_ds,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0,
    collate_fn=data_collator
)

print("Train dataset:", len(train_ds))
print("Dev dataset:", len(dev_ds))
print("Batches - train:", len(train_loader), "dev:", len(dev_loader))

Train dataset: 24429
Dev dataset: 6736
Batches - train: 1527 dev: 421


In [7]:
class RobertaCustomNLI(nn.Module):
    def __init__(self, model_name: str, num_labels: int, dropout: float = 0.2, hidden_dim_1: int = 256, hidden_dim_2: int = 128):
        super().__init__()
        self.encoder = RobertaModel.from_pretrained(model_name)
        hidden_size = self.encoder.config.hidden_size

        self.classifier = nn.Sequential(
            nn.Linear(hidden_size, hidden_dim_1),
            nn.LayerNorm(hidden_dim_1),
            nn.GELU(),
            nn.Dropout(dropout),

            nn.Linear(hidden_dim_1, hidden_dim_2),
            nn.LayerNorm(hidden_dim_2),
            nn.GELU(),
            nn.Dropout(dropout),

            nn.Linear(hidden_dim_2, num_labels)
        )

    def forward(self, input_ids, attention_mask):
        outputs = self.encoder(
            input_ids=input_ids,
            attention_mask=attention_mask
        )

        cls_repr = outputs.last_hidden_state[:, 0, :]   # first token representation
        logits = self.classifier(cls_repr)
        return logits


model = RobertaCustomNLI(
    model_name=MODEL_NAME,
    num_labels=num_labels,
    dropout=DROPOUT,
    hidden_dim_1=HIDDEN_DIM_1,
    hidden_dim_2=HIDDEN_DIM_2
)

model.to(device)
print(model)

model.safetensors:   0%|          | 0.00/499M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

RobertaModel LOAD REPORT from: roberta-base
Key                             | Status     | 
--------------------------------+------------+-
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
pooler.dense.weight             | MISSING    | 
pooler.dense.bias               | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


RobertaCustomNLI(
  (encoder): RobertaModel(
    (embeddings): RobertaEmbeddings(
      (word_embeddings): Embedding(50265, 768, padding_idx=1)
      (token_type_embeddings): Embedding(1, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
      (position_embeddings): Embedding(514, 768, padding_idx=1)
    )
    (encoder): RobertaEncoder(
      (layer): ModuleList(
        (0-11): 12 x RobertaLayer(
          (attention): RobertaAttention(
            (self): RobertaSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): RobertaSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): La

In [8]:
from torch.optim import AdamW

criterion = nn.CrossEntropyLoss()

optimizer = AdamW(
    model.parameters(),
    lr=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY
)

total_steps = len(train_loader) * EPOCHS
warmup_steps = int(total_steps * WARMUP_RATIO)

scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=warmup_steps,
    num_training_steps=total_steps
)

print("Total steps:", total_steps)
print("Warmup steps:", warmup_steps)

Total steps: 4581
Warmup steps: 274


In [9]:
@torch.no_grad()
def evaluate(model, dataloader, criterion):
    model.eval()

    all_preds = []
    all_labels = []
    total_loss = 0.0

    for batch in tqdm(dataloader, desc="Evaluating", leave=False):
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)

        logits = model(
            input_ids=input_ids,
            attention_mask=attention_mask
        )

        loss = criterion(logits, labels)
        total_loss += loss.item()

        preds = torch.argmax(logits, dim=-1)

        all_preds.extend(preds.cpu().numpy().tolist())
        all_labels.extend(labels.cpu().numpy().tolist())

    avg_loss = total_loss / max(1, len(dataloader))
    acc = accuracy_score(all_labels, all_preds)
    f1_macro = f1_score(all_labels, all_preds, average="macro")

    return avg_loss, acc, f1_macro, all_labels, all_preds

In [10]:
def train_one_epoch(model, dataloader, optimizer, scheduler, criterion, epoch_idx: int):
    model.train()
    running_loss = 0.0

    for batch in tqdm(dataloader, desc=f"Training epoch {epoch_idx+1}", leave=False):
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)

        optimizer.zero_grad(set_to_none=True)

        logits = model(
            input_ids=input_ids,
            attention_mask=attention_mask
        )

        loss = criterion(logits, labels)
        loss.backward()

        torch.nn.utils.clip_grad_norm_(model.parameters(), MAX_GRAD_NORM)

        optimizer.step()
        scheduler.step()

        running_loss += loss.item()

    avg_loss = running_loss / max(1, len(dataloader))
    return avg_loss

In [11]:
best_f1 = -1.0
best_acc = -1.0
history = []

for epoch in range(EPOCHS):
    train_loss = train_one_epoch(
        model=model,
        dataloader=train_loader,
        optimizer=optimizer,
        scheduler=scheduler,
        criterion=criterion,
        epoch_idx=epoch
    )

    dev_loss, dev_acc, dev_f1, dev_labels, dev_preds = evaluate(
        model=model,
        dataloader=dev_loader,
        criterion=criterion
    )

    print(
        f"Epoch {epoch+1}/{EPOCHS} | "
        f"Train Loss: {train_loss:.4f} | "
        f"Dev Loss: {dev_loss:.4f} | "
        f"Dev Acc: {dev_acc:.4f} | "
        f"Dev Macro-F1: {dev_f1:.4f}"
    )

    history.append({
        "epoch": epoch + 1,
        "train_loss": train_loss,
        "dev_loss": dev_loss,
        "dev_acc": dev_acc,
        "dev_macro_f1": dev_f1
    })

    if dev_f1 > best_f1:
        best_f1 = dev_f1
        best_acc = dev_acc

        print("New best model found. Saving to:", OUTPUT_DIR)

        # Save weights
        torch.save(model.state_dict(), os.path.join(OUTPUT_DIR, "model_state_dict.pt"))

        # Save tokenizer
        tokenizer.save_pretrained(OUTPUT_DIR)

        # Save training history
        with open(os.path.join(OUTPUT_DIR, "history.json"), "w") as f:
            json.dump(history, f, indent=2)

        # Save metadata needed to rebuild model later
        metadata = {
            "model_name": MODEL_NAME,
            "num_labels": num_labels,
            "max_length": MAX_LENGTH,
            "batch_size": BATCH_SIZE,
            "epochs_trained": epoch + 1,
            "learning_rate": LEARNING_RATE,
            "weight_decay": WEIGHT_DECAY,
            "warmup_ratio": WARMUP_RATIO,
            "max_grad_norm": MAX_GRAD_NORM,
            "dropout": DROPOUT,
            "hidden_dim_1": HIDDEN_DIM_1,
            "hidden_dim_2": HIDDEN_DIM_2,
            "best_dev_acc": best_acc,
            "best_dev_macro_f1": best_f1,
            "label2id": label2id,
            "id2label": id2label,
            "premise_col": premise_col,
            "hypothesis_col": hypothesis_col,
            "label_col": label_col
        }

        with open(os.path.join(OUTPUT_DIR, "training_metadata.json"), "w") as f:
            json.dump(metadata, f, indent=2)

print("\nBest Dev Acc:", best_acc)
print("Best Dev Macro-F1:", best_f1)

Training epoch 1:   0%|          | 0/1527 [00:00<?, ?it/s]

Evaluating:   0%|          | 0/421 [00:00<?, ?it/s]

Epoch 1/3 | Train Loss: 0.4472 | Dev Loss: 0.3197 | Dev Acc: 0.8689 | Dev Macro-F1: 0.8689
New best model found. Saving to: roberta_custom_head_nli


Training epoch 2:   0%|          | 0/1527 [00:00<?, ?it/s]

Evaluating:   0%|          | 0/421 [00:00<?, ?it/s]

Epoch 2/3 | Train Loss: 0.2505 | Dev Loss: 0.4390 | Dev Acc: 0.8792 | Dev Macro-F1: 0.8789
New best model found. Saving to: roberta_custom_head_nli


Training epoch 3:   0%|          | 0/1527 [00:00<?, ?it/s]

Evaluating:   0%|          | 0/421 [00:00<?, ?it/s]

Epoch 3/3 | Train Loss: 0.1679 | Dev Loss: 0.4936 | Dev Acc: 0.8849 | Dev Macro-F1: 0.8848
New best model found. Saving to: roberta_custom_head_nli

Best Dev Acc: 0.8849465558194775
Best Dev Macro-F1: 0.8847735626355036


In [12]:
# Reload best model weights into a fresh model instance (optional but clean)
best_model = RobertaCustomNLI(
    model_name=MODEL_NAME,
    num_labels=num_labels,
    dropout=DROPOUT,
    hidden_dim_1=HIDDEN_DIM_1,
    hidden_dim_2=HIDDEN_DIM_2
)

best_model.load_state_dict(torch.load(os.path.join(OUTPUT_DIR, "model_state_dict.pt"), map_location=device))
best_model.to(device)

dev_loss, dev_acc, dev_f1, dev_labels, dev_preds = evaluate(
    model=best_model,
    dataloader=dev_loader,
    criterion=criterion
)

print(f"Final Best Dev Loss: {dev_loss:.4f}")
print(f"Final Best Dev Acc: {dev_acc:.4f}")
print(f"Final Best Dev Macro-F1: {dev_f1:.4f}")

print("\nClassification Report:")
print(classification_report(dev_labels, dev_preds))

print("\nConfusion Matrix:")
print(confusion_matrix(dev_labels, dev_preds))

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

RobertaModel LOAD REPORT from: roberta-base
Key                             | Status     | 
--------------------------------+------------+-
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
pooler.dense.weight             | MISSING    | 
pooler.dense.bias               | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Evaluating:   0%|          | 0/421 [00:00<?, ?it/s]

Final Best Dev Loss: 0.4936
Final Best Dev Acc: 0.8849
Final Best Dev Macro-F1: 0.8848

Classification Report:
              precision    recall  f1-score   support

           0       0.89      0.87      0.88      3258
           1       0.88      0.89      0.89      3478

    accuracy                           0.88      6736
   macro avg       0.88      0.88      0.88      6736
weighted avg       0.88      0.88      0.88      6736


Confusion Matrix:
[[2850  408]
 [ 367 3111]]


bi encoder model

In [13]:
BI_OUTPUT_DIR = "roberta_bi_encoder_nli"
ENSEMBLE_OUTPUT_DIR = "roberta_hybrid_ensemble"
os.makedirs(BI_OUTPUT_DIR, exist_ok=True)
os.makedirs(ENSEMBLE_OUTPUT_DIR, exist_ok=True)

print("Bi-encoder output dir:", BI_OUTPUT_DIR)
print("Ensemble output dir:", ENSEMBLE_OUTPUT_DIR)

Bi-encoder output dir: roberta_bi_encoder_nli
Ensemble output dir: roberta_hybrid_ensemble


In [14]:
class NLIBiEncoderDataset(Dataset):
    def __init__(self, df: pd.DataFrame, tokenizer, premise_col: str, hypothesis_col: str, label_col: str, max_length: int):
        self.df = df.reset_index(drop=True)
        self.tokenizer = tokenizer
        self.premise_col = premise_col
        self.hypothesis_col = hypothesis_col
        self.label_col = label_col
        self.max_length = max_length

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx: int):
        row = self.df.iloc[idx]
        prem = row[self.premise_col]
        hyp = row[self.hypothesis_col]
        label = int(row[self.label_col])

        prem_enc = self.tokenizer(
            prem,
            truncation=True,
            max_length=self.max_length,
            padding=False,
            return_tensors=None,
        )

        hyp_enc = self.tokenizer(
            hyp,
            truncation=True,
            max_length=self.max_length,
            padding=False,
            return_tensors=None,
        )

        return {
            "premise_input_ids": prem_enc["input_ids"],
            "premise_attention_mask": prem_enc["attention_mask"],
            "hypothesis_input_ids": hyp_enc["input_ids"],
            "hypothesis_attention_mask": hyp_enc["attention_mask"],
            "labels": label
        }

In [15]:
def bi_collate_fn(features):
    premise_features = [
        {
            "input_ids": f["premise_input_ids"],
            "attention_mask": f["premise_attention_mask"]
        }
        for f in features
    ]

    hypothesis_features = [
        {
            "input_ids": f["hypothesis_input_ids"],
            "attention_mask": f["hypothesis_attention_mask"]
        }
        for f in features
    ]

    premise_batch = tokenizer.pad(
        premise_features,
        padding=True,
        return_tensors="pt"
    )

    hypothesis_batch = tokenizer.pad(
        hypothesis_features,
        padding=True,
        return_tensors="pt"
    )

    labels = torch.tensor([f["labels"] for f in features], dtype=torch.long)

    batch = {
        "premise_input_ids": premise_batch["input_ids"],
        "premise_attention_mask": premise_batch["attention_mask"],
        "hypothesis_input_ids": hypothesis_batch["input_ids"],
        "hypothesis_attention_mask": hypothesis_batch["attention_mask"],
        "labels": labels
    }
    return batch

In [16]:
train_bi_ds = NLIBiEncoderDataset(
    train_df,
    tokenizer,
    premise_col=premise_col,
    hypothesis_col=hypothesis_col,
    label_col="label_id",
    max_length=MAX_LENGTH
)

dev_bi_ds = NLIBiEncoderDataset(
    dev_df,
    tokenizer,
    premise_col=premise_col,
    hypothesis_col=hypothesis_col,
    label_col="label_id",
    max_length=MAX_LENGTH
)

train_bi_loader = DataLoader(
    train_bi_ds,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=0,
    collate_fn=bi_collate_fn
)

dev_bi_loader = DataLoader(
    dev_bi_ds,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0,
    collate_fn=bi_collate_fn
)

print("Bi-encoder train dataset:", len(train_bi_ds))
print("Bi-encoder dev dataset:", len(dev_bi_ds))
print("Bi-encoder batches - train:", len(train_bi_loader), "dev:", len(dev_bi_loader))

Bi-encoder train dataset: 24429
Bi-encoder dev dataset: 6736
Bi-encoder batches - train: 1527 dev: 421


In [17]:
def mean_pooling(last_hidden_state, attention_mask):
    mask = attention_mask.unsqueeze(-1).expand(last_hidden_state.size()).float()
    masked_embeddings = last_hidden_state * mask
    summed = masked_embeddings.sum(dim=1)
    counts = mask.sum(dim=1).clamp(min=1e-9)
    return summed / counts

In [18]:
class RobertaBiEncoderNLI(nn.Module):
    def __init__(self, model_name: str, num_labels: int, dropout: float = 0.2, hidden_dim_1: int = 256, hidden_dim_2: int = 128):
        super().__init__()
        self.encoder = RobertaModel.from_pretrained(model_name)
        hidden_size = self.encoder.config.hidden_size

        combined_size = hidden_size * 3  # u, v, |u-v|

        self.classifier = nn.Sequential(
            nn.Linear(combined_size, hidden_dim_1),
            nn.LayerNorm(hidden_dim_1),
            nn.GELU(),
            nn.Dropout(dropout),

            nn.Linear(hidden_dim_1, hidden_dim_2),
            nn.LayerNorm(hidden_dim_2),
            nn.GELU(),
            nn.Dropout(dropout),

            nn.Linear(hidden_dim_2, num_labels)
        )

    def forward(
        self,
        premise_input_ids,
        premise_attention_mask,
        hypothesis_input_ids,
        hypothesis_attention_mask
    ):
        premise_outputs = self.encoder(
            input_ids=premise_input_ids,
            attention_mask=premise_attention_mask
        )
        hypothesis_outputs = self.encoder(
            input_ids=hypothesis_input_ids,
            attention_mask=hypothesis_attention_mask
        )

        u = mean_pooling(premise_outputs.last_hidden_state, premise_attention_mask)
        v = mean_pooling(hypothesis_outputs.last_hidden_state, hypothesis_attention_mask)

        features = torch.cat([u, v, torch.abs(u - v)], dim=-1)
        logits = self.classifier(features)
        return logits


bi_model = RobertaBiEncoderNLI(
    model_name=MODEL_NAME,
    num_labels=num_labels,
    dropout=DROPOUT,
    hidden_dim_1=HIDDEN_DIM_1,
    hidden_dim_2=HIDDEN_DIM_2
)

bi_model.to(device)
print(bi_model)

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

RobertaModel LOAD REPORT from: roberta-base
Key                             | Status     | 
--------------------------------+------------+-
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
pooler.dense.weight             | MISSING    | 
pooler.dense.bias               | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


RobertaBiEncoderNLI(
  (encoder): RobertaModel(
    (embeddings): RobertaEmbeddings(
      (word_embeddings): Embedding(50265, 768, padding_idx=1)
      (token_type_embeddings): Embedding(1, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
      (position_embeddings): Embedding(514, 768, padding_idx=1)
    )
    (encoder): RobertaEncoder(
      (layer): ModuleList(
        (0-11): 12 x RobertaLayer(
          (attention): RobertaAttention(
            (self): RobertaSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): RobertaSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm):

In [19]:
bi_criterion = nn.CrossEntropyLoss()

bi_optimizer = AdamW(
    bi_model.parameters(),
    lr=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY
)

bi_total_steps = len(train_bi_loader) * EPOCHS
bi_warmup_steps = int(bi_total_steps * WARMUP_RATIO)

bi_scheduler = get_linear_schedule_with_warmup(
    bi_optimizer,
    num_warmup_steps=bi_warmup_steps,
    num_training_steps=bi_total_steps
)

print("Bi total steps:", bi_total_steps)
print("Bi warmup steps:", bi_warmup_steps)

Bi total steps: 4581
Bi warmup steps: 274


In [20]:
@torch.no_grad()
def evaluate_bi(model, dataloader, criterion):
    model.eval()

    all_preds = []
    all_labels = []
    total_loss = 0.0

    for batch in tqdm(dataloader, desc="Evaluating Bi-Encoder", leave=False):
        premise_input_ids = batch["premise_input_ids"].to(device)
        premise_attention_mask = batch["premise_attention_mask"].to(device)
        hypothesis_input_ids = batch["hypothesis_input_ids"].to(device)
        hypothesis_attention_mask = batch["hypothesis_attention_mask"].to(device)
        labels = batch["labels"].to(device)

        logits = model(
            premise_input_ids=premise_input_ids,
            premise_attention_mask=premise_attention_mask,
            hypothesis_input_ids=hypothesis_input_ids,
            hypothesis_attention_mask=hypothesis_attention_mask
        )

        loss = criterion(logits, labels)
        total_loss += loss.item()

        preds = torch.argmax(logits, dim=-1)

        all_preds.extend(preds.cpu().numpy().tolist())
        all_labels.extend(labels.cpu().numpy().tolist())

    avg_loss = total_loss / max(1, len(dataloader))
    acc = accuracy_score(all_labels, all_preds)
    f1_macro = f1_score(all_labels, all_preds, average="macro")

    return avg_loss, acc, f1_macro, all_labels, all_preds

In [21]:
def train_one_epoch_bi(model, dataloader, optimizer, scheduler, criterion, epoch_idx: int):
    model.train()
    running_loss = 0.0

    for batch in tqdm(dataloader, desc=f"Training Bi-Encoder epoch {epoch_idx+1}", leave=False):
        premise_input_ids = batch["premise_input_ids"].to(device)
        premise_attention_mask = batch["premise_attention_mask"].to(device)
        hypothesis_input_ids = batch["hypothesis_input_ids"].to(device)
        hypothesis_attention_mask = batch["hypothesis_attention_mask"].to(device)
        labels = batch["labels"].to(device)

        optimizer.zero_grad(set_to_none=True)

        logits = model(
            premise_input_ids=premise_input_ids,
            premise_attention_mask=premise_attention_mask,
            hypothesis_input_ids=hypothesis_input_ids,
            hypothesis_attention_mask=hypothesis_attention_mask
        )

        loss = criterion(logits, labels)
        loss.backward()

        torch.nn.utils.clip_grad_norm_(model.parameters(), MAX_GRAD_NORM)

        optimizer.step()
        scheduler.step()

        running_loss += loss.item()

    avg_loss = running_loss / max(1, len(dataloader))
    return avg_loss

In [22]:
best_bi_f1 = -1.0
best_bi_acc = -1.0
bi_history = []

for epoch in range(EPOCHS):
    bi_train_loss = train_one_epoch_bi(
        model=bi_model,
        dataloader=train_bi_loader,
        optimizer=bi_optimizer,
        scheduler=bi_scheduler,
        criterion=bi_criterion,
        epoch_idx=epoch
    )

    bi_dev_loss, bi_dev_acc, bi_dev_f1, bi_dev_labels, bi_dev_preds = evaluate_bi(
        model=bi_model,
        dataloader=dev_bi_loader,
        criterion=bi_criterion
    )

    print(
        f"[Bi-Encoder] Epoch {epoch+1}/{EPOCHS} | "
        f"Train Loss: {bi_train_loss:.4f} | "
        f"Dev Loss: {bi_dev_loss:.4f} | "
        f"Dev Acc: {bi_dev_acc:.4f} | "
        f"Dev Macro-F1: {bi_dev_f1:.4f}"
    )

    bi_history.append({
        "epoch": epoch + 1,
        "train_loss": bi_train_loss,
        "dev_loss": bi_dev_loss,
        "dev_acc": bi_dev_acc,
        "dev_macro_f1": bi_dev_f1
    })

    if bi_dev_f1 > best_bi_f1:
        best_bi_f1 = bi_dev_f1
        best_bi_acc = bi_dev_acc

        print("New best bi-encoder model found. Saving to:", BI_OUTPUT_DIR)

        torch.save(bi_model.state_dict(), os.path.join(BI_OUTPUT_DIR, "model_state_dict.pt"))
        tokenizer.save_pretrained(BI_OUTPUT_DIR)

        with open(os.path.join(BI_OUTPUT_DIR, "history.json"), "w") as f:
            json.dump(bi_history, f, indent=2)

        bi_metadata = {
            "model_name": MODEL_NAME,
            "num_labels": num_labels,
            "max_length": MAX_LENGTH,
            "batch_size": BATCH_SIZE,
            "epochs_trained": epoch + 1,
            "learning_rate": LEARNING_RATE,
            "weight_decay": WEIGHT_DECAY,
            "warmup_ratio": WARMUP_RATIO,
            "max_grad_norm": MAX_GRAD_NORM,
            "dropout": DROPOUT,
            "hidden_dim_1": HIDDEN_DIM_1,
            "hidden_dim_2": HIDDEN_DIM_2,
            "best_dev_acc": best_bi_acc,
            "best_dev_macro_f1": best_bi_f1
        }

        with open(os.path.join(BI_OUTPUT_DIR, "training_metadata.json"), "w") as f:
            json.dump(bi_metadata, f, indent=2)

print("\nBest Bi-Encoder Dev Acc:", best_bi_acc)
print("Best Bi-Encoder Dev Macro-F1:", best_bi_f1)

Training Bi-Encoder epoch 1:   0%|          | 0/1527 [00:00<?, ?it/s]

Evaluating Bi-Encoder:   0%|          | 0/421 [00:00<?, ?it/s]

[Bi-Encoder] Epoch 1/3 | Train Loss: 0.5590 | Dev Loss: 0.4901 | Dev Acc: 0.7683 | Dev Macro-F1: 0.7682
New best bi-encoder model found. Saving to: roberta_bi_encoder_nli


Training Bi-Encoder epoch 2:   0%|          | 0/1527 [00:00<?, ?it/s]

Evaluating Bi-Encoder:   0%|          | 0/421 [00:00<?, ?it/s]

[Bi-Encoder] Epoch 2/3 | Train Loss: 0.4219 | Dev Loss: 0.4746 | Dev Acc: 0.7840 | Dev Macro-F1: 0.7832
New best bi-encoder model found. Saving to: roberta_bi_encoder_nli


Training Bi-Encoder epoch 3:   0%|          | 0/1527 [00:00<?, ?it/s]

Evaluating Bi-Encoder:   0%|          | 0/421 [00:00<?, ?it/s]

[Bi-Encoder] Epoch 3/3 | Train Loss: 0.3104 | Dev Loss: 0.5326 | Dev Acc: 0.7870 | Dev Macro-F1: 0.7864
New best bi-encoder model found. Saving to: roberta_bi_encoder_nli

Best Bi-Encoder Dev Acc: 0.7869655581947743
Best Bi-Encoder Dev Macro-F1: 0.7863788942305068


In [23]:
best_bi_model = RobertaBiEncoderNLI(
    model_name=MODEL_NAME,
    num_labels=num_labels,
    dropout=DROPOUT,
    hidden_dim_1=HIDDEN_DIM_1,
    hidden_dim_2=HIDDEN_DIM_2
)

best_bi_model.load_state_dict(torch.load(os.path.join(BI_OUTPUT_DIR, "model_state_dict.pt"), map_location=device))
best_bi_model.to(device)

bi_dev_loss, bi_dev_acc, bi_dev_f1, bi_dev_labels, bi_dev_preds = evaluate_bi(
    model=best_bi_model,
    dataloader=dev_bi_loader,
    criterion=bi_criterion
)

print(f"Final Best Bi-Encoder Dev Loss: {bi_dev_loss:.4f}")
print(f"Final Best Bi-Encoder Dev Acc: {bi_dev_acc:.4f}")
print(f"Final Best Bi-Encoder Dev Macro-F1: {bi_dev_f1:.4f}")

print("\nBi-Encoder Classification Report:")
print(classification_report(bi_dev_labels, bi_dev_preds))

print("\nBi-Encoder Confusion Matrix:")
print(confusion_matrix(bi_dev_labels, bi_dev_preds))

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

RobertaModel LOAD REPORT from: roberta-base
Key                             | Status     | 
--------------------------------+------------+-
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
pooler.dense.weight             | MISSING    | 
pooler.dense.bias               | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Evaluating Bi-Encoder:   0%|          | 0/421 [00:00<?, ?it/s]

Final Best Bi-Encoder Dev Loss: 0.5326
Final Best Bi-Encoder Dev Acc: 0.7870
Final Best Bi-Encoder Dev Macro-F1: 0.7864

Bi-Encoder Classification Report:
              precision    recall  f1-score   support

           0       0.79      0.76      0.78      3258
           1       0.78      0.81      0.80      3478

    accuracy                           0.79      6736
   macro avg       0.79      0.79      0.79      6736
weighted avg       0.79      0.79      0.79      6736


Bi-Encoder Confusion Matrix:
[[2474  784]
 [ 651 2827]]


In [24]:
@torch.no_grad()
def get_cross_logits(model, dataloader):
    model.eval()
    all_logits = []
    all_labels = []

    for batch in tqdm(dataloader, desc="Cross logits", leave=False):
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)

        logits = model(input_ids=input_ids, attention_mask=attention_mask)

        all_logits.append(logits.cpu())
        all_labels.append(labels.cpu())

    all_logits = torch.cat(all_logits, dim=0)
    all_labels = torch.cat(all_labels, dim=0)
    return all_logits, all_labels


@torch.no_grad()
def get_bi_logits(model, dataloader):
    model.eval()
    all_logits = []
    all_labels = []

    for batch in tqdm(dataloader, desc="Bi logits", leave=False):
        premise_input_ids = batch["premise_input_ids"].to(device)
        premise_attention_mask = batch["premise_attention_mask"].to(device)
        hypothesis_input_ids = batch["hypothesis_input_ids"].to(device)
        hypothesis_attention_mask = batch["hypothesis_attention_mask"].to(device)
        labels = batch["labels"].to(device)

        logits = model(
            premise_input_ids=premise_input_ids,
            premise_attention_mask=premise_attention_mask,
            hypothesis_input_ids=hypothesis_input_ids,
            hypothesis_attention_mask=hypothesis_attention_mask
        )

        all_logits.append(logits.cpu())
        all_labels.append(labels.cpu())

    all_logits = torch.cat(all_logits, dim=0)
    all_labels = torch.cat(all_labels, dim=0)
    return all_logits, all_labels


cross_logits_dev, cross_labels_dev = get_cross_logits(best_model, dev_loader)
bi_logits_dev, bi_labels_dev = get_bi_logits(best_bi_model, dev_bi_loader)

assert torch.equal(cross_labels_dev, bi_labels_dev), "Label order mismatch between cross and bi dev loaders."

dev_labels_np = cross_labels_dev.numpy()
print("Cross logits shape:", cross_logits_dev.shape)
print("Bi logits shape:", bi_logits_dev.shape)

Cross logits:   0%|          | 0/421 [00:00<?, ?it/s]

Bi logits:   0%|          | 0/421 [00:00<?, ?it/s]

Cross logits shape: torch.Size([6736, 2])
Bi logits shape: torch.Size([6736, 2])


In [25]:
def evaluate_weighted_ensemble(cross_logits, bi_logits, labels, alpha):
    final_logits = alpha * cross_logits + (1.0 - alpha) * bi_logits
    preds = torch.argmax(final_logits, dim=-1).numpy()

    acc = accuracy_score(labels, preds)
    f1_macro = f1_score(labels, preds, average="macro")
    return acc, f1_macro, preds

alpha_results = []

for alpha in np.arange(0.0, 1.01, 0.1):
    acc, f1_macro, preds = evaluate_weighted_ensemble(
        cross_logits=cross_logits_dev,
        bi_logits=bi_logits_dev,
        labels=dev_labels_np,
        alpha=float(alpha)
    )
    alpha_results.append({
        "alpha": round(float(alpha), 2),
        "dev_acc": acc,
        "dev_macro_f1": f1_macro
    })

alpha_results_df = pd.DataFrame(alpha_results)
display(alpha_results_df.sort_values("dev_macro_f1", ascending=False))

best_alpha_row = alpha_results_df.sort_values("dev_macro_f1", ascending=False).iloc[0]
best_alpha = float(best_alpha_row["alpha"])

print("Best alpha:", best_alpha)
print("Best ensemble dev acc:", best_alpha_row["dev_acc"])
print("Best ensemble dev macro F1:", best_alpha_row["dev_macro_f1"])

,alpha,dev_acc,dev_macro_f1
7,0.7,0.886134,0.885944
5,0.5,0.885986,0.885797
6,0.6,0.885986,0.885794
8,0.8,0.885837,0.885655
9,0.9,0.885095,0.884921
10,1.0,0.884947,0.884774
4,0.4,0.877375,0.877112
3,0.3,0.853771,0.853368
2,0.2,0.832096,0.831511
1,0.1,0.806562,0.805921


Best alpha: 0.7
Best ensemble dev acc: 0.8861342042755345
Best ensemble dev macro F1: 0.8859441056551834


In [26]:
ensemble_dev_acc, ensemble_dev_f1, ensemble_dev_preds = evaluate_weighted_ensemble(
    cross_logits=cross_logits_dev,
    bi_logits=bi_logits_dev,
    labels=dev_labels_np,
    alpha=best_alpha
)

print(f"Final Ensemble Dev Acc: {ensemble_dev_acc:.4f}")
print(f"Final Ensemble Dev Macro-F1: {ensemble_dev_f1:.4f}")

print("\nEnsemble Classification Report:")
print(classification_report(dev_labels_np, ensemble_dev_preds))

print("\nEnsemble Confusion Matrix:")
print(confusion_matrix(dev_labels_np, ensemble_dev_preds))

ensemble_metadata = {
    "best_alpha": best_alpha,
    "dev_acc": ensemble_dev_acc,
    "dev_macro_f1": ensemble_dev_f1
}

with open(os.path.join(ENSEMBLE_OUTPUT_DIR, "ensemble_metadata.json"), "w") as f:
    json.dump(ensemble_metadata, f, indent=2)

Final Ensemble Dev Acc: 0.8861
Final Ensemble Dev Macro-F1: 0.8859

Ensemble Classification Report:
              precision    recall  f1-score   support

           0       0.89      0.87      0.88      3258
           1       0.88      0.90      0.89      3478

    accuracy                           0.89      6736
   macro avg       0.89      0.89      0.89      6736
weighted avg       0.89      0.89      0.89      6736


Ensemble Confusion Matrix:
[[2847  411]
 [ 356 3122]]


In [27]:
import numpy as np

# Cross-encoder predictions
np.savetxt("cross_encoder_predictions.txt", dev_preds, fmt="%d")

# Bi-encoder predictions
np.savetxt("bi_encoder_predictions.txt", bi_dev_preds, fmt="%d")

# Ensemble predictions
np.savetxt("ensemble_predictions.txt", ensemble_dev_preds, fmt="%d")

print("All prediction files saved.")

All prediction files saved.
